# Ingest Pfam Annotation GFF → `nmdc_results.pfam_annotation_gff` (issue #88)

Loads `loaded_pfam_gff/pfam_annotation_gff.parquet` (output of `parse_pfam_gff.ipynb`)
into the `nmdc` tenant's lakehouse as `nmdc_results.pfam_annotation_gff`.

Two-phase flow (matching BERIL convention):
1. **Upload** the local Parquet to MinIO bronze under
   `tenant-general-warehouse/nmdc/datasets/results/`.
2. **Ingest** registers the Delta table under
   `s3a://cdm-lake/tenant-sql-warehouse/nmdc/nmdc_results.db/pfam_annotation_gff`.

After ingest, query as e.g.
`SELECT * FROM nmdc_results.pfam_annotation_gff LIMIT 10`.

### Configuration

In [1]:
import json
import os
import time
from pathlib import Path

TENANT  = "nmdc"
DATASET = "results"
BUCKET  = "cdm-lake"

TABLE_NAME = "pfam_annotation_gff"
FORCE_OVERWRITE = False  # safety: refuse to clobber existing table unless True

SOURCE_DIR    = Path(os.environ.get("PFAM_GFF_OUT_DIR", "loaded_pfam_gff"))
LOCAL_PARQUET = SOURCE_DIR / f"{TABLE_NAME}.parquet"
if not LOCAL_PARQUET.exists():
    raise RuntimeError(
        f"Source Parquet not found: {LOCAL_PARQUET.resolve()}. "
        "Run parse_pfam_gff.ipynb first."
    )

NAMESPACE     = f"{TENANT}_{DATASET}"
BRONZE_PREFIX = f"tenant-general-warehouse/{TENANT}/datasets/{DATASET}"
BRONZE_BASE   = f"s3a://{BUCKET}/{BRONZE_PREFIX}"
SILVER_BASE   = f"s3a://{BUCKET}/tenant-sql-warehouse/{TENANT}/{NAMESPACE}.db"

size_gb = LOCAL_PARQUET.stat().st_size / 1024**3
print(f"Tenant         : {TENANT}")
print(f"Namespace      : {NAMESPACE}")
print(f"Table          : {NAMESPACE}.{TABLE_NAME}")
print(f"Source parquet : {LOCAL_PARQUET.resolve()}  ({size_gb:.2f} GB)")
print(f"Bronze base    : {BRONZE_BASE}")
print(f"Silver base    : {SILVER_BASE}")
print(f"Force overwrite: {FORCE_OVERWRITE}")

Tenant         : nmdc
Namespace      : nmdc_results
Table          : nmdc_results.pfam_annotation_gff
Source parquet : /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_pfam_gff/pfam_annotation_gff.parquet  (48.71 GB)
Bronze base    : s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/results
Silver base    : s3a://cdm-lake/tenant-sql-warehouse/nmdc/nmdc_results.db
Force overwrite: False


### Spark + MinIO setup

> **BERDL kernel note:** `get_spark_session()`, `get_minio_client()`, and `ingest()` are auto-imported by the BERDL JupyterHub kernel startup script. If you run this notebook outside BERDL, import them yourself or substitute equivalents to avoid `NameError`.

In [2]:
spark = get_spark_session(app_name=f"ingest_{NAMESPACE}_{TABLE_NAME}", tenant_name=TENANT)
mc    = get_minio_client()
print(f"Spark version: {spark.version}")
try:
    print(f"MinIO endpoint: {mc._base_url._url.netloc}")
except AttributeError:
    print("MinIO endpoint: (not directly inspectable)")

Spark version: 4.0.1
MinIO endpoint: minio.berdl.kbase.us


### Preflight: skip if already registered

`pfam_annotation_gff` is multi-hour to load. Refuse to silently re-overwrite
an existing copy unless the user explicitly opts in via `FORCE_OVERWRITE`.

In [3]:
if spark.catalog.databaseExists(NAMESPACE):
    registered = {
        row.tableName for row in spark.sql(f"SHOW TABLES IN {NAMESPACE}").collect()
    }
else:
    registered = set()
    print(f"Namespace {NAMESPACE} does not exist yet — first-time load.")

if TABLE_NAME in registered and not FORCE_OVERWRITE:
    raise RuntimeError(
        f"{NAMESPACE}.{TABLE_NAME} is already registered. "
        "Set FORCE_OVERWRITE = True in the config cell to re-overwrite."
    )
if TABLE_NAME in registered:
    print(f"FORCE_OVERWRITE: will replace existing {NAMESPACE}.{TABLE_NAME}")

### Phase 1 — upload the Parquet to MinIO bronze

Spark executors run on other cluster nodes and can't see local files, so we
upload the Parquet via the driver-side MinIO client. The Parquet is ~50 GB
(compressed from ~650 GB of raw GFFs); upload runs in 10-15 min at MinIO
LAN speeds.

In [4]:
object_key = f"{BRONZE_PREFIX}/{TABLE_NAME}.parquet"
bronze_uri = f"s3a://{BUCKET}/{object_key}"

size_gb = LOCAL_PARQUET.stat().st_size / 1024**3
print(f"Uploading {LOCAL_PARQUET.name} ({size_gb:.2f} GB) → {bronze_uri}")
t0 = time.monotonic()
mc.fput_object(BUCKET, object_key, str(LOCAL_PARQUET))
elapsed = time.monotonic() - t0
print(f"Upload complete in {elapsed/60:.1f} min ({size_gb*1024/elapsed:.0f} MB/s)")

Uploading pfam_annotation_gff.parquet (48.71 GB) → s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/results/pfam_annotation_gff.parquet
Upload complete in 13.9 min (60 MB/s)


### Phase 2 — build the ingest config

In [5]:
config = {
    "tenant":  TENANT,
    "dataset": DATASET,
    "paths": {
        "bronze_base": BRONZE_BASE,
        "silver_base": SILVER_BASE,
    },
    "defaults": {
        "parquet": {},
    },
    "tables": [
        {
            "name":        TABLE_NAME,
            "format":      "parquet",
            "bronze_path": f"{TABLE_NAME}.parquet",
            "write_mode":  "overwrite",
        }
    ],
}
print(json.dumps(config, indent=2))

{
  "tenant": "nmdc",
  "dataset": "results",
  "paths": {
    "bronze_base": "s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/results",
    "silver_base": "s3a://cdm-lake/tenant-sql-warehouse/nmdc/nmdc_results.db"
  },
  "defaults": {
    "parquet": {}
  },
  "tables": [
    {
      "name": "pfam_annotation_gff",
      "format": "parquet",
      "bronze_path": "pfam_annotation_gff.parquet",
      "write_mode": "overwrite"
    }
  ]
}


### Phase 3 — run ingest

`ingest()` reads the bronze Parquet and writes a Delta table. At ~3 B rows
this is the slowest cell — expect tens of minutes.

In [6]:
report = ingest(config=config, spark=spark, minio_client=mc)
print(json.dumps(report, indent=2, default=str))

{"time": "2026-05-02 17:45:52,457", "pipeline": "data_lakehouse_ingest", "schema": "default", "table": "pipeline_stage", "level": "INFO", "module": "config_loader", "msg": "Using inline dict configuration"}
{"time": "2026-05-02 17:45:52,458", "pipeline": "data_lakehouse_ingest", "schema": "default", "table": "pipeline_stage", "level": "INFO", "module": "config_loader", "msg": "Config loaded successfully"}
{"time": "2026-05-02 17:45:52,458", "pipeline": "data_lakehouse_ingest", "schema": "default", "table": "pipeline_stage", "level": "INFO", "module": "config_loader", "msg": "Table 'pfam_annotation_gff' has no explicit schema ('schema_sql'/'schema'); schema will be inferred by the loader."}
{"time": "2026-05-02 17:45:52,459", "pipeline": "data_lakehouse_ingest", "schema": "default", "table": "pipeline_stage", "level": "INFO", "module": "config_loader", "msg": "Minimal config validation passed"}
{"time": "2026-05-02 17:45:52,459", "pipeline": "data_lakehouse_ingest", "schema": "default",

### Smoke test

In [7]:
n_rows = spark.sql(f"SELECT COUNT(*) FROM {NAMESPACE}.{TABLE_NAME}").collect()[0][0]
print(f"Row count: {n_rows:,}")

sample = spark.sql(f"""
    SELECT workflow_run_id, gene_id, pfam_accession, start, end, score, e_value
    FROM {NAMESPACE}.{TABLE_NAME}
    LIMIT 10
""").toPandas()
display(sample)

Row count: 2,684,369,000


,workflow_run_id,gene_id,pfam_accession,start,end,score,e_value
0,NaN,nmdc:wfmgan-11-6k38sx51.1_034382_161_559,PF06723,18,133,183.0,6.600000e-53
1,NaN,nmdc:wfmgan-11-6k38sx51.1_034383_3_557,PF07973,39,94,63.5,1.600000e-16
2,NaN,nmdc:wfmgan-11-6k38sx51.1_034384_2_337,PF04386,18,112,129.0,1.600000e-36
3,NaN,nmdc:wfmgan-11-6k38sx51.1_034387_2_559,PF08207,40,97,75.5,2.600000e-20
4,NaN,nmdc:wfmgan-11-6k38sx51.1_034387_2_559,PF01132,105,158,69.9,1.500000e-18
5,NaN,nmdc:wfmgan-11-6k38sx51.1_034388_2_559,PF00550,143,186,27.5,3.100000e-05
6,NaN,nmdc:wfmgan-11-6k38sx51.1_034390_1_276,PF02768,1,90,77.5,7.700000e-21
7,NaN,nmdc:wfmgan-11-6k38sx51.1_034392_2_559,PF01087,1,145,182.0,1.600000e-52
8,NaN,nmdc:wfmgan-11-6k38sx51.1_034393_69_557,PF02769,19,157,46.0,6.100000e-11
9,NaN,nmdc:wfmgan-11-6k38sx51.1_034397_1_552,PF11987,1,72,83.9,8.500000e-23


In [ ]:
# Demo: join to nmdc_ref_data.pfam_terms — skip cleanly if pfam_terms isn't loaded
# so a missing prerequisite doesn't look like a failed ingest.
ref_tables = set()
if spark.catalog.databaseExists("nmdc_ref_data"):
    ref_tables = {r.tableName for r in spark.sql("SHOW TABLES IN nmdc_ref_data").collect()}

if "pfam_terms" not in ref_tables:
    print("nmdc_ref_data.pfam_terms is not registered — skipping join demo. "
          "Load it via notebooks/load_pfam_terms.ipynb if you want this query.")
else:
    demo = spark.sql(f"""
        SELECT p.pfam_accession, t.name, t.description, COUNT(*) AS n_hits
        FROM {NAMESPACE}.{TABLE_NAME} p
        JOIN nmdc_ref_data.pfam_terms t ON t.pfam_id = p.pfam_accession
        WHERE p.pfam_accession IN ('PF04183', 'PF06276')
        GROUP BY p.pfam_accession, t.name, t.description
        ORDER BY n_hits DESC
    """).toPandas()
    print("Siderophore/iron-reductase domain hits across all workflow runs:")
    display(demo)